# 14 — project adapter validation（保存済み30シナリオ）

## 背景
Notebook 13は4秒のequation-level proxyで、robot modelを動かす証拠ではない。
project所有adapterで過去に保存されたUnitree A1 MuJoCo plantの各scenarioについて、
定量metricとGIF playback時間の存在・整合だけを読む。このNotebookは再実行しない。

## 目的
1. `src/legged_control_mujoco` のA1 adapterについて保存済み30条件の証拠を読む。
2. easy/normal/hardが各10件、simulation/GIFが各20秒以上か機械検証する。
3. pass/failだけでなく、失敗理由と物理metricを追って調整箇所へ戻る。

## 厳密な実装境界
- 上流正本: `external/legged_control/` commit `a7f381c0367e98e31c01336e678eef47e304d40d`。ROS1、OCS2 SQP-NMPC、
  Pinocchio/qpOASES WBC、Gazebo/Unitree I/Oの原実装。
- 実行対象: project所有 `src/legged_control_mujoco/adapter.py` と `models/a1.xml`。
- adapterはgait template、24D state/input contract、WBC task構造、hybrid torque式を対応させるが、
  **OCS2 SQPではない**。有限horizon policyを、瞬時friction-constrained force plannerと
  MuJoCo acceleration-level inverse dynamicsで置換する。
- したがって結果は「project adapterの保存済み挙動」であり、上流ROS1/OCS2 repository性能ではない。
- estimatorとGazebo/Unitree hardware pathはこのadapter実行経路に存在しない。
- ROS2 portは作成・compile・実行されておらず、この結果はROS2検証に一切使えない。
- この経路は **Quadruped-PyMPCを一切使用しない**。

## 結論
保存されたmetricとGIFが揃ったscenarioだけを実行済みとみなす。閾値passはadapterについての
保存結果に対する判定であり、OCS2 SQP、元のWBC、実機A1、ROS2 parityの主張へ外挿しない。


## 検証範囲に関する必須注記

このprojectでは **ROS2 portを作成・compile・実行していない**。したがってROS2 parityは
**NOT VERIFIED / FAIL-CLOSED** である。上流commit `a7f381c0367e98e31c01336e678eef47e304d40d` はROS1実装であり、
project所有MuJoCo adapterはOCS2のhorizon SQPを瞬時force plannerへ、
Pinocchio/qpOASES WBCをMuJoCo acceleration inverse dynamicsへ置換し、
元のestimator/hardware経路も持たない。保存済み30 scenario dataが示すのはadapter挙動だけで、
上流 `legged_control` の性能でもROS2移行の検証でもない。


In [ ]:
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`from pathlib import Path` の依存を明示して再現可能な実行環境を作る。
from pathlib import Path
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`import numpy as np` の依存を明示して再現可能な実行環境を作る。
import numpy as np
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`import matplotlib.pyplot as plt` の依存を明示して再現可能な実行環境を作る。
import matplotlib.pyplot as plt

# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = Path.cwd()` の演算・変換をPythonで評価する。
ROOT = Path.cwd()
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`for candidate in [ROOT, *ROOT.parents]:` の反復範囲を固定して各sampleを処理する。 数式: `for candidate in [ROOT, *ROOT.parents]:` の演算・変換をPythonで評価する。
for candidate in [ROOT, *ROOT.parents]:
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`if (candidate / "pyproject.toml").exists():` の条件で安全側の実行分岐を選ぶ。 数式: `if (candidate / "pyproject.toml").exists():` の演算・変換をPythonで評価する。
    if (candidate / "pyproject.toml").exists():
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = candidate` の演算・変換をPythonで評価する。
        ROOT = candidate
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`break` をこの章の処理順に沿って実行する。
        break

# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`np.set_printoptions(precision` を後続計算で使う明示的な中間量として設定する。 数式: `np.set_printoptions(precision=4, suppress=True)` の演算・変換をPythonで評価する。
np.set_printoptions(precision=4, suppress=True)
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})` の要素または終端を対応付ける。
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`print("repository:", ROOT)` の観測値を表示して判定根拠を残す。
print("repository:", ROOT)

# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`import csv` の依存を明示して再現可能な実行環境を作る。
import csv
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`import json` の依存を明示して再現可能な実行環境を作る。
import json
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`from PIL import Image` の依存を明示して再現可能な実行環境を作る。
from PIL import Image

# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`SCENARIO_ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `SCENARIO_ROOT = ROOT / "notebook_legged" / "assets" / "scenarios"` の演算・変換をPythonで評価する。
SCENARIO_ROOT = ROOT / "notebook_legged" / "assets" / "scenarios"
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`JSON_PATH` を後続計算で使う明示的な中間量として設定する。 数式: `JSON_PATH = SCENARIO_ROOT / "scenario_results.json"` の演算・変換をPythonで評価する。
JSON_PATH = SCENARIO_ROOT / "scenario_results.json"
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`CSV_PATH` を後続計算で使う明示的な中間量として設定する。 数式: `CSV_PATH = SCENARIO_ROOT / "scenario_results.csv"` の演算・変換をPythonで評価する。
CSV_PATH = SCENARIO_ROOT / "scenario_results.csv"


## ASCIIデータフロー
```text
scenario(name,gait,command,friction,payload,push,seed)
  -> A1 MuJoCo model (q,v,contacts,M,b,J)
  -> instantaneous force planner
       min ||W(J^T f - (M qdd* + b))||² + lambda||f-f_nom||²
       fz>=0, |fx|<=mu*fz, |fy|<=mu*fz
  -> acceleration WBC
       stance/swing Cartesian qdd* + posture regularization
       M qdd + b - J^T f - S^T tau = 0
  -> hybrid torque
       tau_cmd=clip(tau+0(q*-q)+3(dq*-dq), +/-33.5)
  -> MuJoCo plant -- q,v,measured contact --> next control sample
  -> metrics + 20 s GIF -> JSON/CSV/gallery
```

## source / symbol / equation mapping
```cpp
// upstream: external/legged_control/legged_controllers/config/a1/gait.info
ModeSchedule::from_gait             // adapter.py: GAIT_TEMPLATES, mode_phase
// upstream: centroidal input first 12 entries
A1HeadlessAdapter::_optimize_contact_forces
  demand=(M*qdd+b)[0:6]             // floating-base wrench equation
  minimize ||W(J^T f-demand)||^2    // instantaneous; NOT OCS2 SQP
// upstream: WbcBase::formulateNoContactMotionTask/formulateSwingLegTask
A1HeadlessAdapter::_desired_qacc
  J*qdd = a_foot^* - Jdot*qdot      // stance/swing acceleration task
// upstream: WbcBase.cpp floating-base EoM and torque extraction
A1HeadlessAdapter::solve_wbc
  tau=(M*qdd+b-J^T*f)[actuated]      // then +/-33.5 N m
// upstream: LeggedController.cpp setCommand(...,0,3,tau)
adapter.py::hybrid_command           // ff + Kp error + Kd error
// project runner
scripts/run_legged_control_benchmark.py::run_scenario/write_aggregates
```


## 判定閾値
benchmark scriptの `Thresholds` が正本:

- simulation duration ≥ 20 s、GIF playback ≥ 20 s
- fallなし、minimum base height ≥ 0.18 m
- height RMSE ≤ 0.10 m、max |roll/pitch| ≤ 0.60 rad
- planar velocity RMSE ≤ 0.35 m/s、yaw-rate RMSE ≤ 0.60 rad/s
- max torque ≤ 33.5 N m、saturation fraction ≤ 0.10
- post-saturation dynamics residual ≤ 5.0
- planned/measured contact agreement ≥ 0.55

転倒検出自体はheight < 0.18 mまたは|roll/pitch| > 0.9 rad。判定姿勢閾値0.60 radの方が厳しい。


In [ ]:
# --- Block 1: JSON/CSVを同時にloadし、30/10/10と名前集合を検証 ---
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`EXPECTED` を後続計算で使う明示的な中間量として設定する。 数式: `EXPECTED = [('easy', 'E01_stance_baseline'), ('easy', 'E02_stance_low'),…` の演算・変換をPythonで評価する。
EXPECTED = [('easy', 'E01_stance_baseline'), ('easy', 'E02_stance_low'), ('easy', 'E03_stance_high'), ('easy', 'E04_walk_005'), ('easy', 'E05_walk_008'), ('easy', 'E06_walk_lateral'), ('easy', 'E07_stance_payload'), ('easy', 'E08_stance_gentle_push'), ('easy', 'E09_walk_turn'), ('easy', 'E10_walk_012'), ('normal', 'N01_walk_016'), ('normal', 'N02_walk_diagonal'), ('normal', 'N03_walk_turn'), ('normal', 'N04_dynamic_walk'), ('normal', 'N05_standing_trot'), ('normal', 'N06_trot'), ('normal', 'N07_walk_payload'), ('normal', 'N08_walk_push'), ('normal', 'N09_walk_mu045'), ('normal', 'N10_walk_low_turn_push'), ('hard', 'H01_walk_025'), ('hard', 'H02_walk_strafe'), ('hard', 'H03_walk_fast_turn'), ('hard', 'H04_trot_fast'), ('hard', 'H05_flying_trot'), ('hard', 'H06_pace'), ('hard', 'H07_low_friction'), ('hard', 'H08_heavy_payload'), ('hard', 'H09_strong_push'), ('hard', 'H10_compound')]
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`expected_names` を後続計算で使う明示的な中間量として設定する。 数式: `expected_names = [name for _, name in EXPECTED]` の演算・変換をPythonで評価する。
expected_names = [name for _, name in EXPECTED]
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`expected_levels` を後続計算で使う明示的な中間量として設定する。 数式: `expected_levels = {level: sum(item[0] == level for item in EXPECTED)` の演算・変換をPythonで評価する。
expected_levels = {level: sum(item[0] == level for item in EXPECTED)
                   # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`for level in ("easy", "normal", "hard")}` の反復範囲を固定して各sampleを処理する。
                   for level in ("easy", "normal", "hard")}
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`assert len(EXPECTED) == 30 and expected_levels == {"easy": 10, "normal":…` を不変条件として即時検査する。 数式: `assert len(EXPECTED) == 30 and expected_levels == {"easy": 10, "normal":…` の演算・変換をPythonで評価する。
assert len(EXPECTED) == 30 and expected_levels == {"easy": 10, "normal": 10, "hard": 10}

# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`if not JSON_PATH.is_file() or not CSV_PATH.is_file():` の条件で安全側の実行分岐を選ぶ。
if not JSON_PATH.is_file() or not CSV_PATH.is_file():
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`print("PENDING: run uv run python scripts/run_legged_control_benchmark.p…` の観測値を表示して判定根拠を残す。 数式: `print("PENDING: run uv run python scripts/run_legged_control_benchmark.p…` の演算・変換をPythonで評価する。
    print("PENDING: run uv run python scripts/run_legged_control_benchmark.py --all")
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`records, csv_rows` を後続計算で使う明示的な中間量として設定する。 数式: `records, csv_rows = [], []` の演算・変換をPythonで評価する。
    records, csv_rows = [], []
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`else:` の条件で安全側の実行分岐を選ぶ。
else:
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`aggregate` を後続計算で使う明示的な中間量として設定する。 数式: `aggregate = json.loads(JSON_PATH.read_text(encoding="utf-8"))` の演算・変換をPythonで評価する。
    aggregate = json.loads(JSON_PATH.read_text(encoding="utf-8"))
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`records` を後続計算で使う明示的な中間量として設定する。 数式: `records = aggregate["results"]` の演算・変換をPythonで評価する。
    records = aggregate["results"]
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`with CSV_PATH.open(newline="", encoding="utf-8") as handle:` のresource境界を閉じ忘れなく扱う。 数式: `with CSV_PATH.open(newline="", encoding="utf-8") as handle:` の演算・変換をPythonで評価する。
    with CSV_PATH.open(newline="", encoding="utf-8") as handle:
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`csv_rows` を後続計算で使う明示的な中間量として設定する。 数式: `csv_rows = list(csv.DictReader(handle))` の演算・変換をPythonで評価する。
        csv_rows = list(csv.DictReader(handle))
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`json_names` を後続計算で使う明示的な中間量として設定する。 数式: `json_names = [record["config"]["name"] for record in records]` の演算・変換をPythonで評価する。
    json_names = [record["config"]["name"] for record in records]
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`csv_names` を後続計算で使う明示的な中間量として設定する。 数式: `csv_names = [row["name"] for row in csv_rows]` の演算・変換をPythonで評価する。
    csv_names = [row["name"] for row in csv_rows]
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`assert len(records) == len(csv_rows) == 30` を不変条件として即時検査する。 数式: `assert len(records) == len(csv_rows) == 30` の演算・変換をPythonで評価する。
    assert len(records) == len(csv_rows) == 30
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`assert set(json_names) == set(csv_names) == set(expected_names)` を不変条件として即時検査する。 数式: `assert set(json_names) == set(csv_names) == set(expected_names)` の演算・変換をPythonで評価する。
    assert set(json_names) == set(csv_names) == set(expected_names)
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`counts` を後続計算で使う明示的な中間量として設定する。 数式: `counts = {level: sum(r["config"]["difficulty"] == level for r in records…` の演算・変換をPythonで評価する。
    counts = {level: sum(r["config"]["difficulty"] == level for r in records)
              # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`for level in ("easy", "normal", "hard")}` の反復範囲を固定して各sampleを処理する。
              for level in ("easy", "normal", "hard")}
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`assert counts == {"easy": 10, "normal": 10, "hard": 10}` を不変条件として即時検査する。 数式: `assert counts == {"easy": 10, "normal": 10, "hard": 10}` の演算・変換をPythonで評価する。
    assert counts == {"easy": 10, "normal": 10, "hard": 10}
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`print("validated scenario counts:", counts)` の観測値を表示して判定根拠を残す。
    print("validated scenario counts:", counts)


In [ ]:
# --- Block 2: Pillowで全30 GIFのframe timingをdecodeして20秒以上を検証 ---
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`gif_playback_seconds` の責務を独立関数として定義する。
def gif_playback_seconds(path):
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`total_ms` を後続計算で使う明示的な中間量として設定する。 数式: `total_ms = 0` の演算・変換をPythonで評価する。
    total_ms = 0
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`with Image.open(path) as image:` のresource境界を閉じ忘れなく扱う。
    with Image.open(path) as image:
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`frame_count` を後続計算で使う明示的な中間量として設定する。 数式: `frame_count = image.n_frames` の演算・変換をPythonで評価する。
        frame_count = image.n_frames
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`for index in range(frame_count):` の反復範囲を固定して各sampleを処理する。
        for index in range(frame_count):
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `image.seek(index)` の要素または終端を対応付ける。
            image.seek(index)
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`total_ms +` を後続計算で使う明示的な中間量として設定する。 数式: `total_ms += int(image.info.get("duration", 0))` の演算・変換をPythonで評価する。
            total_ms += int(image.info.get("duration", 0))
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`return frame_count, total_ms / 1000.0` の値を次の制御境界へ返す。 数式: `return frame_count, total_ms / 1000.0` の演算・変換をPythonで評価する。
    return frame_count, total_ms / 1000.0

# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`gif_checks` を後続計算で使う明示的な中間量として設定する。 数式: `gif_checks = []` の演算・変換をPythonで評価する。
gif_checks = []
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`for level, name in EXPECTED:` の反復範囲を固定して各sampleを処理する。
for level, name in EXPECTED:
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`path` を後続計算で使う明示的な中間量として設定する。 数式: `path = SCENARIO_ROOT / "gifs" / f"{name}.gif"` の演算・変換をPythonで評価する。
    path = SCENARIO_ROOT / "gifs" / f"{name}.gif"
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`assert path.is_file(), f"missing GIF: {path}"` を不変条件として即時検査する。
    assert path.is_file(), f"missing GIF: {path}"
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`frames, playback_s` を後続計算で使う明示的な中間量として設定する。 数式: `frames, playback_s = gif_playback_seconds(path)` の演算・変換をPythonで評価する。
    frames, playback_s = gif_playback_seconds(path)
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`assert frames > 0, f"empty GIF: {name}"` を不変条件として即時検査する。
    assert frames > 0, f"empty GIF: {name}"
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`assert playback_s + 1e-9 >= 20.0, f"{name}: GIF playback {playback_s:.3f…` を不変条件として即時検査する。 数式: `assert playback_s + 1e-9 >= 20.0, f"{name}: GIF playback {playback_s:.3f…` の演算・変換をPythonで評価する。
    assert playback_s + 1e-9 >= 20.0, f"{name}: GIF playback {playback_s:.3f} s < 20 s"
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `gif_checks.append((level, name, frames, playback_s))` の要素または終端を対応付ける。
    gif_checks.append((level, name, frames, playback_s))
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`print(f"validated {len(gif_checks)} GIFs; minimum playback:",` の観測値を表示して判定根拠を残す。
print(f"validated {len(gif_checks)} GIFs; minimum playback:",
      # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `min(item[3] for item in gif_checks), "s")` の要素または終端を対応付ける。
      min(item[3] for item in gif_checks), "s")


In [ ]:
# --- Block 3: 保存metricを再表示し、失敗理由を隠さない ---
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`if records:` の条件で安全側の実行分岐を選ぶ。
if records:
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`summary` を後続計算で使う明示的な中間量として設定する。 数式: `summary = []` の演算・変換をPythonで評価する。
    summary = []
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`for record in records:` の反復範囲を固定して各sampleを処理する。
    for record in records:
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`cfg, metric` を後続計算で使う明示的な中間量として設定する。 数式: `cfg, metric = record["config"], record["metrics"]` の演算・変換をPythonで評価する。
        cfg, metric = record["config"], record["metrics"]
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`summary.append({` をこの章の処理順に沿って実行する。
        summary.append({
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"name": cfg["name"], "difficulty": cfg["difficulty"],` の要素または終端を対応付ける。
            "name": cfg["name"], "difficulty": cfg["difficulty"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"passed": metric["passed"],` の要素または終端を対応付ける。
            "passed": metric["passed"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"sim_s": metric["simulated_duration_s"],` の要素または終端を対応付ける。
            "sim_s": metric["simulated_duration_s"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"gif_s": metric["gif_playback_duration_s"],` の要素または終端を対応付ける。
            "gif_s": metric["gif_playback_duration_s"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"height_rmse_m": metric["height_error_rmse_m"],` の要素または終端を対応付ける。
            "height_rmse_m": metric["height_error_rmse_m"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"velocity_rmse_mps": metric["velocity_tracking_rmse"]["planar_mps"],` の要素または終端を対応付ける。
            "velocity_rmse_mps": metric["velocity_tracking_rmse"]["planar_mps"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"max_rp_rad": metric["maximum_abs_roll_pitch_rad"],` の要素または終端を対応付ける。
            "max_rp_rad": metric["maximum_abs_roll_pitch_rad"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"max_tau_nm": metric["maximum_abs_torque_nm"],` の要素または終端を対応付ける。
            "max_tau_nm": metric["maximum_abs_torque_nm"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"dyn_residual": metric["maximum_dynamics_residual"],` の要素または終端を対応付ける。
            "dyn_residual": metric["maximum_dynamics_residual"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"contact_agreement": metric["planned_vs_measured_contact_agreement"],` の要素または終端を対応付ける。
            "contact_agreement": metric["planned_vs_measured_contact_agreement"],
            # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `"failure_reasons": "; ".join(metric["failure_reasons"]) or "none",` の要素または終端を対応付ける。
            "failure_reasons": "; ".join(metric["failure_reasons"]) or "none",
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `})` の要素または終端を対応付ける。
        })
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`import pandas as pd` の依存を明示して再現可能な実行環境を作る。
    import pandas as pd
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`result_df` を後続計算で使う明示的な中間量として設定する。 数式: `result_df = pd.DataFrame(summary).sort_values("name")` の演算・変換をPythonで評価する。
    result_df = pd.DataFrame(summary).sort_values("name")
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`display(result_df)` の観測値を表示して判定根拠を残す。
    display(result_df)
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`display(result_df.groupby("difficulty").agg(` の観測値を表示して判定根拠を残す。
    display(result_df.groupby("difficulty").agg(
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`scenarios` を後続計算で使う明示的な中間量として設定する。 数式: `scenarios=("name", "count"), passed=("passed", "sum"),` の演算・変換をPythonで評価する。
        scenarios=("name", "count"), passed=("passed", "sum"),
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`mean_velocity_rmse` を後続計算で使う明示的な中間量として設定する。 数式: `mean_velocity_rmse=("velocity_rmse_mps", "mean"),` の演算・変換をPythonで評価する。
        mean_velocity_rmse=("velocity_rmse_mps", "mean"),
        # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`worst_dynamics_residual` を後続計算で使う明示的な中間量として設定する。 数式: `worst_dynamics_residual=("dyn_residual", "max"),` の演算・変換をPythonで評価する。
        worst_dynamics_residual=("dyn_residual", "max"),
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、直前の式・構造へ `))` の要素または終端を対応付ける。
    ))
# 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`else:` の条件で安全側の実行分岐を選ぶ。
else:
    # 背景: project adapterは上流のOCS2/WBCを別ロジックへ置換する。目的: 保存済みadapter結果の完全性だけを検査するため、`print("PENDING: aggregate files are not complete yet")` の観測値を表示して判定根拠を残す。
    print("PENDING: aggregate files are not complete yet")


## 生成時の測定結果
集約済み `30/30`、pass `5`、fail `25`。閾値判定はbenchmark scriptの保存値を表示する。

- `E01_stance_baseline` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E02_stance_low` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E03_stance_high` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E04_walk_005` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `E05_walk_008` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `E06_walk_lateral` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `E07_stance_payload` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E08_stance_gentle_push` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E09_walk_turn` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; dynamics residual exceeds threshold
- `E10_walk_012` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold
- `N01_walk_016` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `N02_walk_diagonal` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; dynamics residual exceeds threshold
- `N03_walk_turn` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `N04_dynamic_walk` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold
- `N05_standing_trot` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `N06_trot` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `N07_walk_payload` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `N08_walk_push` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `N09_walk_mu045` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold
- `N10_walk_low_turn_push` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H01_walk_025` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H02_walk_strafe` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H03_walk_fast_turn` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H04_trot_fast` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H05_flying_trot` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H06_pace` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H07_low_friction` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H08_heavy_payload` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H09_strong_push` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H10_compound` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold


## Easy 10


### `E01_stance_baseline`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E01_stance_baseline](assets/scenarios/gifs/E01_stance_baseline.gif)


### `E02_stance_low`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E02_stance_low](assets/scenarios/gifs/E02_stance_low.gif)


### `E03_stance_high`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E03_stance_high](assets/scenarios/gifs/E03_stance_high.gif)


### `E04_walk_005`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E04_walk_005](assets/scenarios/gifs/E04_walk_005.gif)


### `E05_walk_008`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E05_walk_008](assets/scenarios/gifs/E05_walk_008.gif)


### `E06_walk_lateral`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E06_walk_lateral](assets/scenarios/gifs/E06_walk_lateral.gif)


### `E07_stance_payload`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E07_stance_payload](assets/scenarios/gifs/E07_stance_payload.gif)


### `E08_stance_gentle_push`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E08_stance_gentle_push](assets/scenarios/gifs/E08_stance_gentle_push.gif)


### `E09_walk_turn`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E09_walk_turn](assets/scenarios/gifs/E09_walk_turn.gif)


### `E10_walk_012`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E10_walk_012](assets/scenarios/gifs/E10_walk_012.gif)


## Normal 10


### `N01_walk_016`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N01_walk_016](assets/scenarios/gifs/N01_walk_016.gif)


### `N02_walk_diagonal`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N02_walk_diagonal](assets/scenarios/gifs/N02_walk_diagonal.gif)


### `N03_walk_turn`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N03_walk_turn](assets/scenarios/gifs/N03_walk_turn.gif)


### `N04_dynamic_walk`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N04_dynamic_walk](assets/scenarios/gifs/N04_dynamic_walk.gif)


### `N05_standing_trot`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N05_standing_trot](assets/scenarios/gifs/N05_standing_trot.gif)


### `N06_trot`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N06_trot](assets/scenarios/gifs/N06_trot.gif)


### `N07_walk_payload`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N07_walk_payload](assets/scenarios/gifs/N07_walk_payload.gif)


### `N08_walk_push`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N08_walk_push](assets/scenarios/gifs/N08_walk_push.gif)


### `N09_walk_mu045`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N09_walk_mu045](assets/scenarios/gifs/N09_walk_mu045.gif)


### `N10_walk_low_turn_push`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N10_walk_low_turn_push](assets/scenarios/gifs/N10_walk_low_turn_push.gif)


## Hard 10


### `H01_walk_025`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H01_walk_025](assets/scenarios/gifs/H01_walk_025.gif)


### `H02_walk_strafe`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H02_walk_strafe](assets/scenarios/gifs/H02_walk_strafe.gif)


### `H03_walk_fast_turn`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H03_walk_fast_turn](assets/scenarios/gifs/H03_walk_fast_turn.gif)


### `H04_trot_fast`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H04_trot_fast](assets/scenarios/gifs/H04_trot_fast.gif)


### `H05_flying_trot`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H05_flying_trot](assets/scenarios/gifs/H05_flying_trot.gif)


### `H06_pace`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H06_pace](assets/scenarios/gifs/H06_pace.gif)


### `H07_low_friction`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H07_low_friction](assets/scenarios/gifs/H07_low_friction.gif)


### `H08_heavy_payload`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H08_heavy_payload](assets/scenarios/gifs/H08_heavy_payload.gif)


### `H09_strong_push`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H09_strong_push](assets/scenarios/gifs/H09_strong_push.gif)


### `H10_compound`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H10_compound](assets/scenarios/gifs/H10_compound.gif)
